# HandSense AI
**Digital Aiding 4 Aging Hackathon 2026**

รันทีละ Cell จากบนลงล่าง

## Cell 1: ติดตั้ง packages
> หลังรัน Cell นี้เสร็จ ให้กด **Runtime → Restart session** แล้วรัน Cell 2-3 ต่อ

In [ ]:
!pip install mediapipe==0.10.35 streamlit opencv-python-headless numpy pandas -q
print('done - please Restart Runtime now (Runtime menu -> Restart session)')

## Cell 2: เขียน app.py

In [ ]:
%%writefile app.py
"""
HandSense AI - Learned Non-Use Detection
Digital Aiding 4 Aging Hackathon 2026 | AI Vibe Coding
"""

import sys
import types

# Block mediapipe.tasks from loading TensorFlow (protobuf conflict in Colab)
for _m in ['mediapipe.tasks', 'mediapipe.tasks.python',
           'mediapipe.tasks.python.audio', 'mediapipe.tasks.python.components',
           'mediapipe.tasks.python.core', 'mediapipe.tasks.python.text',
           'mediapipe.tasks.python.vision']:
    sys.modules[_m] = types.ModuleType(_m)

import streamlit as st
import cv2
import numpy as np
import tempfile
import os

from mediapipe.python.solutions.hands import Hands


# --- Metric Functions ---

def calc_speed(positions):
    if len(positions) < 2:
        return 0.0
    dists = [np.linalg.norm(np.array(positions[i]) - np.array(positions[i-1]))
             for i in range(1, len(positions))]
    return float(np.mean(dists))


def calc_smoothness(positions):
    if len(positions) < 5:
        return 0.0
    p = np.array(positions)
    vel = np.diff(p, axis=0)
    acc = np.diff(vel, axis=0)
    jerk = np.diff(acc, axis=0)
    mean_jerk = np.mean(np.linalg.norm(jerk, axis=1))
    return float(1.0 / (mean_jerk + 1e-6))


def calc_range_of_motion(positions):
    if len(positions) < 2:
        return 0.0
    p = np.array(positions)
    rom = np.max(p, axis=0) - np.min(p, axis=0)
    return float(np.linalg.norm(rom))


def calc_finger_spread(spreads):
    return float(np.mean(spreads)) if spreads else 0.0


# --- Video Analysis ---

def analyze_video(video_path, progress_bar=None):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    data = {
        "Left":  {"positions": [], "spreads": [], "frame_count": 0},
        "Right": {"positions": [], "spreads": [], "frame_count": 0},
    }

    frame_idx = 0
    with Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.55,
        min_tracking_confidence=0.55,
    ) as hands:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            if progress_bar and total_frames > 0:
                progress_bar.progress(min(frame_idx / total_frames, 1.0))

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)

            if result.multi_hand_landmarks and result.multi_handedness:
                for lm, hd in zip(result.multi_hand_landmarks, result.multi_handedness):
                    side = hd.classification[0].label
                    wrist = lm.landmark[0]
                    data[side]["positions"].append((wrist.x, wrist.y))
                    data[side]["frame_count"] += 1
                    t = lm.landmark[4]
                    pk = lm.landmark[20]
                    spread = np.sqrt((t.x - pk.x) ** 2 + (t.y - pk.y) ** 2)
                    data[side]["spreads"].append(spread)

    cap.release()
    return data


def compute_metrics(side_data):
    pos = side_data["positions"]
    spr = side_data["spreads"]
    if len(pos) < 5:
        return None
    speed      = calc_speed(pos)
    smoothness = calc_smoothness(pos)
    rom        = calc_range_of_motion(pos)
    spread     = calc_finger_spread(spr)
    smooth_norm = min(smoothness * speed, speed * 3)
    score = speed * 0.40 + smooth_norm * 0.40 + rom * 0.20
    return {
        "speed": speed, "smoothness": smoothness, "rom": rom,
        "spread": spread, "frames": side_data["frame_count"], "score": score,
    }


def diagnose(left_m, right_m):
    if left_m is None and right_m is None:
        return None, "ไม่พบมือในวิดีโอ"
    if left_m is None:
        return "left", "มือซ้ายไม่ถูกใช้งาน — อาจมีสัญญาณ Learned Non-Use"
    if right_m is None:
        return "right", "มือขวาไม่ถูกใช้งาน — อาจมีสัญญาณ Learned Non-Use"
    ratio = left_m["score"] / (right_m["score"] + 1e-9)
    THRESHOLD = 0.70
    if ratio < THRESHOLD:
        pct = round((1 - ratio) * 100, 1)
        return "left", "มือซ้ายคะแนนต่ำกว่ามือขวา " + str(pct) + "% - แนะนำประเมิน Learned Non-Use ที่มือซ้าย"
    elif ratio > (1 / THRESHOLD):
        pct = round((ratio - 1) * 100, 1)
        return "right", "มือขวาคะแนนต่ำกว่ามือซ้าย " + str(pct) + "% - แนะนำประเมิน Learned Non-Use ที่มือขวา"
    else:
        return None, "การเคลื่อนไหวทั้งสองมือสมดุล — ไม่พบสัญญาณ Learned Non-Use"


# --- Streamlit UI ---

def main():
    st.set_page_config(page_title="HandSense AI", page_icon="🖐️", layout="wide")
    st.title("🖐️ HandSense AI")
    st.subheader("ระบบตรวจจับ Learned Non-Use ในผู้สูงอายุผ่านการวิเคราะห์วิดีโอ")
    st.caption("Digital Aiding 4 Aging Hackathon 2026 - AI Vibe Coding - RMUTL NAN")
    st.divider()

    with st.expander("วิธีใช้งาน"):
        st.markdown(
            "1. ถ่ายวิดีโอให้เห็นมือทั้งสองข้าง 30 วินาที - 2 นาที\n"
            "2. อัปโหลดวิดีโอ (.mp4 / .avi / .mov)\n"
            "3. กด วิเคราะห์วิดีโอ\n"
            "4. ดูผลเปรียบเทียบมือซ้าย-ขวาและรายงาน"
        )

    uploaded = st.file_uploader("อัปโหลดวิดีโอ", type=["mp4", "avi", "mov", "mkv"])
    if uploaded is None:
        st.info("กรุณาอัปโหลดวิดีโอเพื่อเริ่มการวิเคราะห์")
        st.stop()

    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as tmp:
        tmp.write(uploaded.read())
        tmp_path = tmp.name

    st.video(tmp_path)
    st.divider()

    if st.button("วิเคราะห์วิดีโอ", type="primary", use_container_width=True):
        with st.spinner("กำลังประมวลผล..."):
            prog = st.progress(0.0, text="กำลังอ่านวิดีโอ...")
            raw = analyze_video(tmp_path, progress_bar=prog)
            prog.empty()

        lm = compute_metrics(raw["Left"])
        rm = compute_metrics(raw["Right"])
        suspect, msg = diagnose(lm, rm)

        st.divider()
        st.subheader("ผลการวิเคราะห์")
        if suspect is None and "สมดุล" in msg:
            st.success(msg)
        elif suspect is not None:
            st.warning(msg)
        else:
            st.error(msg)

        st.divider()
        col_l, col_r = st.columns(2)

        def show_metrics(col, label, m, highlight=False):
            with col:
                prefix = "🔴 " if highlight else ""
                st.markdown("### " + prefix + "มือ" + label)
                if m is None:
                    st.error("ไม่พบข้อมูลมือข้างนี้")
                    return
                st.metric("Speed", "{:.4f}".format(m["speed"]))
                st.metric("Smoothness", "{:.2f}".format(m["smoothness"]))
                st.metric("ROM", "{:.4f}".format(m["rom"]))
                st.metric("คะแนนรวม", "{:.4f}".format(m["score"]))
                st.caption("Frames: " + str(m["frames"]))

        show_metrics(col_l, "ซ้าย", lm, highlight=(suspect == "left"))
        show_metrics(col_r, "ขวา",  rm, highlight=(suspect == "right"))

        if lm and rm:
            st.divider()
            st.subheader("เปรียบเทียบคะแนนรวม")
            import pandas as pd
            chart_data = pd.DataFrame({
                "มือ": ["มือซ้าย", "มือขวา"],
                "คะแนน": [lm["score"], rm["score"]],
            }).set_index("มือ")
            st.bar_chart(chart_data)

        st.divider()
        st.subheader("สรุปรายงาน")

        ls  = "{:.4f}".format(lm["speed"])      if lm else "N/A"
        lsm = "{:.4f}".format(lm["smoothness"]) if lm else "N/A"
        lr  = "{:.4f}".format(lm["rom"])         if lm else "N/A"
        lsc = "{:.4f}".format(lm["score"])       if lm else "N/A"
        rs  = "{:.4f}".format(rm["speed"])       if rm else "N/A"
        rsm = "{:.4f}".format(rm["smoothness"])  if rm else "N/A"
        rr  = "{:.4f}".format(rm["rom"])         if rm else "N/A"
        rsc = "{:.4f}".format(rm["score"])       if rm else "N/A"

        report = (
            "**HandSense AI Report**\n\n"
            "| รายการ | มือซ้าย | มือขวา |\n"
            "|--------|---------|--------|\n"
            "| Speed | " + ls + " | " + rs + " |\n"
            "| Smoothness | " + lsm + " | " + rsm + " |\n"
            "| ROM | " + lr + " | " + rr + " |\n"
            "| **คะแนนรวม** | **" + lsc + "** | **" + rsc + "** |\n\n"
            "**สรุป:** " + msg + "\n\n"
            "*วิเคราะห์โดย HandSense AI - Digital Aiding 4 Aging Hackathon 2026*"
        )
        st.markdown(report)

    try:
        os.unlink(tmp_path)
    except Exception:
        pass


if __name__ == "__main__":
    main()


## Cell 3: รัน Streamlit + Public URL

In [ ]:
import threading, time, os
def run(): os.system('streamlit run app.py --server.headless true --server.port 8501')
t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(6)
print('Streamlit started')
!npx localtunnel --port 8501

## วิธีใช้
1. รอจน URL ขึ้น เช่น `https://xxxx.loca.lt`
2. คลิกลิงก์ → อัปโหลดวิดีโอ → กดวิเคราะห์